# Installation and Usage

**1. Clone the Repository**

In [2]:
!git clone https://github.com/Calcifer-0307/Sentiment-Analysis-for-Financial-News.git
%cd Sentiment-Analysis-for-Financial-News

Cloning into 'Sentiment-Analysis-for-Financial-News'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 127 (delta 2), reused 0 (delta 0), pack-reused 118 (from 3)
Receiving objects: 100% (127/127), 62.64 MiB | 21.79 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/Sentiment-Analysis-for-Financial-News


**2. Set Up Virtual Environment**

In [3]:
!python3 -m venv venv
!source venv/bin/activate

Error: Command '['/content/Sentiment-Analysis-for-Financial-News/venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
/bin/bash: line 1: venv/bin/activate: No such file or directory


**3. Install Dependencies**

In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.2 MB/s eta 0:00:00


**4. Download NLTK Data**

In [5]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

**5.Data Preprocessing**

In [6]:
# Ensure your virtual environment is activated
!python src/preprocess.py

Data shape: (4846, 2)
Label distribution:
sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
Text length statistics(character level):
count    4846.000000
mean      128.132068
std        56.526180
min         9.000000
25%        84.000000
50%       119.000000
75%       163.000000
max       315.000000
Name: text, dtype: float64
Text length statistics(word level):
count    4846.000000
mean       23.101114
std         9.958474
min         2.000000
25%        16.000000
50%        21.000000
75%        29.000000
max        81.000000
Name: text, dtype: float64
Data shape: (4846, 3)
Label distribution:
sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
Text length statistics(character level):
count    4846.000000
mean       88.208626
std        43.598506
min         0.000000
25%        55.000000
50%        81.000000
75%       116.000000
max       256.000000
Name: processed_text, dtype: float64
Text length statistics(word l

**6.Loading Data for Training**

In [7]:
from src.data_helper import get_bow_dataloaders, get_tfidf_dataloaders, get_w2c_dataloaders
from src.data_helper import train_bow_or_tfidf_model, train_word2vec_model
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Configuration
train_path = 'data/raw/train_data.csv'
test_path = 'data/raw/test_data.csv'
batch_size = 32

# Example: BoW Model
# 1. Train Vectorizer
bow_model = train_bow_or_tfidf_model([train_path, test_path], CountVectorizer, max_features=5000)

# 2. Get DataLoaders
train_loader, test_loader = get_bow_dataloaders(
    train_path, test_path,
    bow_model,
    batch_size=batch_size
)

# 3. Iterate
for batch_x, batch_y in train_loader:
    # batch_x: (batch_size, 5000)
    # batch_y: (batch_size,)
    pass


# Transformer for Financial News Sentiment Classification

In [24]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import classification_report
import numpy as np
import random

**1. Device & Random Seed**

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
set_seed(42)

Using device: cpu


**2. Load and preprocess data**

In [27]:
train_df = pd.read_csv('data/raw/train_data.csv')
test_df = pd.read_csv('data/raw/test_data.csv')

label_map = {'positive': 0, 'negative': 1, 'neutral': 2}
train_df['label'] = train_df['sentiment'].map(label_map)
test_df['label'] = test_df['sentiment'].map(label_map)

train_df['processed_text'] = train_df['processed_text'].fillna('').astype(str)
test_df['processed_text'] = test_df['processed_text'].fillna('').astype(str)

X_train = train_df['processed_text'].values
y_train = train_df['label'].values
X_test = test_df['processed_text'].values
y_test = test_df['label'].values

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print("Class distribution in training set:", np.bincount(y_train))

Train size: 3634, Test size: 1212
Class distribution in training set: [1022  453 2159]


**3. Build Dataset & DataLoader**

In [28]:
class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        # Use tokenizer directly (replaces deprecated encode_plus)
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Load tokenizer and pre-trained model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
bert_model.to(device)

# Create datasets
train_dataset = BertDataset(X_train, y_train, tokenizer, max_len=64)
test_dataset = BertDataset(X_test, y_test, tokenizer, max_len=64)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**4. Optimizer and Loss**

In [29]:
optimizer = AdamW(bert_model.parameters(), lr=2e-5)
class_weights = torch.tensor([1.0, 2.5, 1.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

**5. Training & Evaluation Functions**

In [30]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, labels)
            total_loss += loss.item()

            _, pred = torch.max(logits, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    return total_loss / len(loader), correct / total

**6. Training Loop**

In [31]:
epochs = 4
for epoch in range(epochs):
    train_loss = train_epoch(bert_model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(bert_model, test_loader, criterion)
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Test Loss={test_loss:.4f}, Test Acc={test_acc:.4f}")

Epoch 1: Train Loss=0.6980, Test Loss=0.5000, Test Acc=0.7913
Epoch 2: Train Loss=0.3903, Test Loss=0.4901, Test Acc=0.7995
Epoch 3: Train Loss=0.2521, Test Loss=0.5325, Test Acc=0.8160
Epoch 4: Train Loss=0.1577, Test Loss=0.6236, Test Acc=0.7995


**7. Final Evaluation & Classification Report**

In [32]:
def get_predictions(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_preds, all_labels

y_pred, y_true = get_predictions(bert_model, test_loader)
target_names = ['positive', 'negative', 'neutral']
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))


Classification Report:
              precision    recall  f1-score   support

    positive       0.79      0.63      0.70       341
    negative       0.62      0.85      0.72       151
     neutral       0.85      0.87      0.86       720

    accuracy                           0.80      1212
   macro avg       0.75      0.78      0.76      1212
weighted avg       0.81      0.80      0.80      1212

